# 08 — Tables & Structured Documents

Tables are not just text with spaces between values.

In a RAG pipeline, a table is a **structured information object** with rows, columns, headers, relationships, and sometimes merged cells or layout-dependent meaning.

If we flatten a table into ordinary text too early, we can destroy the relationships that make the data useful.

In this notebook we will work with the real course documents and examine structured content using:

- DOCX tables with `python-docx`
- PDF tables with PyMuPDF
- PDF → Markdown with PyMuPDF4LLM
- HTML semantic structure with BeautifulSoup
- Markdown tables
- validation and provenance for structured content

The objective is not to force every table into one representation.

The objective is to understand **when structured data should remain structured and when a RAG system needs a textual representation**.

## Learning objectives

By the end of this notebook you should be able to:

1. Detect tables in real documents.
2. Extract DOCX tables without flattening them prematurely.
3. Extract PDF tables with PyMuPDF.
4. Inspect the same PDF table through PyMuPDF4LLM.
5. Convert structured tables into controlled representations.
6. Understand why table-to-text conversion can introduce ambiguity.
7. Preserve table provenance.
8. Validate row/column consistency.
9. Decide when a table belongs in a database, structured payload, or retrieval text.
10. Design a production path for table-heavy documents.

## 1. Install libraries

In [1]:
!pip install -q beautifulsoup4 python-docx pymupdf pymupdf4llm pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. Our real structured-document assets

We will use the files in `data/`:

```text
employee_travel_policy.pdf
q3_customer_support_operating_brief.docx
support_refunds.html
internal_rag_engineering_guide.md
```

The DOCX contains a real service-target table.

The PDF contains a real accommodation table.

The HTML and Markdown documents let us compare other forms of structured content.

In [2]:
from pathlib import Path

DATA_DIR = Path("./data")

PDF_PATH = DATA_DIR / "employee_travel_policy.pdf"
DOCX_PATH = DATA_DIR / "q3_customer_support_operating_brief.docx"
HTML_PATH = DATA_DIR / "support_refunds.html"
MD_PATH = DATA_DIR / "internal_rag_engineering_guide.md"

for path in [PDF_PATH, DOCX_PATH, HTML_PATH, MD_PATH]:
    assert path.exists(), f"Missing: {path}"

print("All source assets found.")

All source assets found.


## 3. Why tables require special treatment

Consider this logical table:

```text
Request type     First response     Target resolution
Refund request   1 business day     7 business days
```

The meaning comes from the **relationship between the header and each cell**.

If the table becomes:

```text
Request type First response Target resolution Refund request 1 business day 7 business days
```

the information is still present, but the relationships are much harder to recover reliably.

That is a retrieval problem.

A model may understand the flattened text, but we should not make the model reconstruct structure that our parser could have preserved.

## 4. DOCX: inspect the document's tables directly

DOCX is particularly useful because tables are represented structurally in the document package.

Use `python-docx` to access the table rather than extracting the entire document into one string.

In [3]:
from docx import Document

docx = Document(DOCX_PATH)

print("Number of tables:", len(docx.tables))

Number of tables: 1


In [4]:
for table_number, table in enumerate(docx.tables, start=1):
    print(f"\nTable {table_number}")
    print("Rows:", len(table.rows))
    print("Columns:", len(table.columns))

    for row in table.rows:
        print([cell.text for cell in row.cells])


Table 1
Rows: 5
Columns: 4
['Request type', 'First response', 'Target resolution', 'Escalation']
['General enquiry', '1 business day', '3 business days', 'After 3 days']
['Refund request', '1 business day', '7 business days', 'After review']
['Payment dispute', '1 business day', '10 business days', 'Finance team']
['Account access', '4 business hours', '1 business day', 'Security team']


This is a strong example of why parser choice matters.

The table's rows and columns are available directly.

We did not need to infer column boundaries from spaces or line breaks.

## 5. Turn the DOCX table into records

A useful structured representation is a list of dictionaries.

We can derive it from the actual table headers.

In [5]:
def docx_table_to_records(table):
    headers = [cell.text.strip() for cell in table.rows[0].cells]

    records = []

    for row in table.rows[1:]:
        values = [cell.text.strip() for cell in row.cells]

        if len(values) != len(headers):
            raise ValueError(
                f"Column mismatch: {len(headers)} headers, {len(values)} values"
            )
        records.append(dict(zip(headers, values)))

    return records

service_table = docx_table_to_records(docx.tables[0])

for record in service_table:
    print(record)

{'Request type': 'General enquiry', 'First response': '1 business day', 'Target resolution': '3 business days', 'Escalation': 'After 3 days'}
{'Request type': 'Refund request', 'First response': '1 business day', 'Target resolution': '7 business days', 'Escalation': 'After review'}
{'Request type': 'Payment dispute', 'First response': '1 business day', 'Target resolution': '10 business days', 'Escalation': 'Finance team'}
{'Request type': 'Account access', 'First response': '4 business hours', 'Target resolution': '1 business day', 'Escalation': 'Security team'}


Now the same real table can be consumed as structured data:

```text
[
    {
        "Request type": "...",
        "First response": "...",
        "Target resolution": "...",
        "Escalation": "..."
    }
]
```

This representation is often preferable when the application needs exact field-level operations.

## 5. Table validation

A production parser should not assume that every row has the same number of cells.

Validate the structure before indexing it.

In [6]:
def validate_table_shape(table):
    if not table.rows:
        return {"rows": 0, "columns": 0}

    expected = len(table.rows[0].cells)

    mismatches = []

    for row_number, row in enumerate(table.rows, start=1):
        actual = len(row.cells)

        if actual != expected:
            mismatches.append({
                "row": row_number,
                "expected_columns": expected,
                "actual_columns": actual,
            })
    
    return {
        "rows": len(table.rows),
        "columns": expected,
        "mismatches": mismatches,
        "valid": not mismatches,
    }

print(validate_table_shape(docx.tables[0]))

{'rows': 5, 'columns': 4, 'mismatches': [], 'valid': True}


## 6. Table text is not always a simple string

Cells may contain:

- multiple paragraphs,
- line breaks,
- formatting,
- lists,
- or nested structures.

Inspect the actual cell paragraphs rather than assuming `cell.text` tells the entire structural story.

In [7]:
table = docx.tables[0]

for row_number, row in enumerate(table.rows, start=1):
    for column_number, cell in enumerate(row.cells, start=1):
        paragraphs = [
            paragraph.text
            for paragraph in cell.paragraphs
        ]

        print(
            f"R{row_number} C{column_number}:",
            paragraphs
        )

R1 C1: ['Request type']
R1 C2: ['First response']
R1 C3: ['Target resolution']
R1 C4: ['Escalation']
R2 C1: ['General enquiry']
R2 C2: ['1 business day']
R2 C3: ['3 business days']
R2 C4: ['After 3 days']
R3 C1: ['Refund request']
R3 C2: ['1 business day']
R3 C3: ['7 business days']
R3 C4: ['After review']
R4 C1: ['Payment dispute']
R4 C2: ['1 business day']
R4 C3: ['10 business days']
R4 C4: ['Finance team']
R5 C1: ['Account access']
R5 C2: ['4 business hours']
R5 C3: ['1 business day']
R5 C4: ['Security team']


The production lesson is:

> A table cell can itself contain structure.

Do not build a parser that assumes every cell is one atomic string unless the corpus guarantees that constraint.

## 8. PDF tables with PyMuPDF

PDFs are different.

A PDF often represents table content through positioned text and drawing objects rather than a native database-like table.

PyMuPDF provides table detection to recover that structure.

In [8]:
import pymupdf

with pymupdf.open(PDF_PATH) as pdf:
    print("Pages:", len(pdf))

    for page_number, page in enumerate(pdf, start=1):
        finder = page.find_tables()
        print(f"Page {page_number}: {len(finder.tables)} table(s)")

Pages: 1
Page 1: 1 table(s)


We are now asking PyMuPDF to perform a structural operation.

This is different from:

```python
page.get_text("text")
```

which produces a textual representation of the page.

## 9. Extract the actual PDF table

Let's inspect the table PyMuPDF detected in the travel policy.

We will not manually construct the expected output; the cells below come directly from the parser.

In [9]:
with pymupdf.open(PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf, start=1):
        finder = page.find_tables()

        for table_number, table in enumerate(finder.tables, start=1):
            print(f"\nPage {page_number}, Table {table_number}")
            rows = table.extract()

            for row in rows:
                print(row)


Page 1, Table 1
['Location', 'Nightly limit', 'Notes']
['Lagos', '₦120,000', 'Standard business accommodation']
['Abuja', '₦110,000', 'Standard business accommodation']
['Port Harcourt', '₦100,000', 'Standard business accommodation']
['International', 'Actual reasonable cost', 'Department approval required']


This is where we can see an important production reality:

**table extraction can be imperfect even when text extraction succeeds.**

A parser can detect the table but still struggle with particular cell boundaries or unusual layouts.

Therefore:

```text
table detected
      ≠
table perfectly reconstructed
```

Table extraction needs validation.

## 10. Inspect the PDF table geometry

PyMuPDF also gives us the table bounding box.

Coordinates are valuable when debugging extraction problems.

In [10]:
with pymupdf.open(PDF_PATH) as pdf:
    page = pdf[0]
    finder = page.find_tables()

    for table in finder.tables:
        print("Table bbox:", table.bbox)
        print("Rows:", table.row_count)
        print("Columns:", table.col_count)

Table bbox: (60.0, 241.99996948242188, 530.0, 366.9999694824219)
Rows: 5
Columns: 3


The bounding box lets us connect structured extraction back to a physical region of the original page.

That is useful for:

- debugging,
- visual verification,
- citation systems,
- document-region classification,
- and extraction quality analysis.

## 11. Table output formats

PyMuPDF table objects can be represented in multiple ways.

Inspect Markdown and pandas representations where available.

In [11]:
with pymupdf.open(PDF_PATH) as pdf:
    page = pdf[0]
    finder = page.find_tables()

    if finder.tables:
        table = finder.tables[0]

        print("--- Markdown ---")
        print(table.to_markdown())

--- Markdown ---
|Location|Nightly limit|Notes|
|---|---|---|
|Lagos|₦120,000|Standard business accommodation|
|Abuja|₦110,000|Standard business accommodation|
|Port Harcourt|₦100,000|Standard business accommodation|
|International|Actual reasonable cost|Department approval required|




In [12]:
with pymupdf.open(PDF_PATH) as pdf:
    page = pdf[0]
    finder = page.find_tables()

    if finder.tables:
        table = finder.tables[0]

        print("--- Markdown ---")
        print(table.to_pandas())

--- Markdown ---
        Location           Nightly limit                            Notes
0          Lagos                ₦120,000  Standard business accommodation
1          Abuja                ₦110,000  Standard business accommodation
2  Port Harcourt                ₦100,000  Standard business accommodation
3  International  Actual reasonable cost     Department approval required


A DataFrame is useful for analysis and validation.

Markdown is useful when a table needs to become part of a text-oriented RAG representation.

The correct representation depends on what happens next.

## 12. PyMuPDF4LLM and tables

Now compare this with the RAG-oriented extraction layer.

PyMuPDF4LLM can turn document structure into Markdown suitable for downstream LLM processing.

In [13]:
import pymupdf4llm

pdf_markdown = pymupdf4llm.to_markdown(PDF_PATH)

print(pdf_markdown[:5000])

# Employee Travel and Reimbursement Policy 

Document ID: HR-TRV-2026-003 | Version: 4.0 | Effective: 1 July 2026 

## 1. Purpose 

This policy establishes the requirements for business travel, eligible expenses, receipts, approvals, and reimbursement. It applies to employees travelling on approved company business. 

## 2. Travel Approval 

Employees must obtain manager approval before booking travel. International travel also requires approval from the relevant department head. 

## 3. Accommodation Limits 

|Location|Nightly limit|Notes|
|---|---|---|
|Lagos|₦120,000|Standard business accommodation|
|Abuja|₦110,000|Standard business accommodation|
|Port Harcourt|₦100,000|Standard business accommodation|
|International|Actual reasonable cost|Department approval required|



## 4. Transportation 

Employees should use reasonable and cost-effective transportation. Air travel should normally be booked in economy class unless an approved exception applies. 

## 5. Receipts and Reimbursem

The important comparison is:

```text
PyMuPDF
    ↓
recover / inspect table structure
    ↓
structured table object

PyMuPDF4LLM
    ↓
RAG-oriented document representation
    ↓
Markdown containing document structure
```

The two layers serve different purposes.

When exact cell-level control matters, keep the structured table.

When the table needs to participate in a text-oriented RAG representation, Markdown can be useful.

## 13. Why table Markdown can still be lossy

Markdown is convenient, but it is still a representation.

A complex table may contain:

- merged cells,
- nested headers,
- footnotes,
- row groups,
- visual alignment,
- multi-line cells,
- or relationships that do not map cleanly to Markdown.

Therefore:

> Converting a table to Markdown does not mean we have preserved every aspect of the original table.

For high-value structured data, retain the structured extraction alongside the RAG representation.

## 14. Keep structured and textual representations together

A production ingestion record can contain both:

```text
table
├── structured representation
│   ├── headers
│   └── rows
│
└── textual representation
    └── Markdown
```

This avoids forcing one representation to serve every downstream system.

In [14]:
with pymupdf.open(PDF_PATH) as pdf:
    page = pdf[0]
    finder = page.find_tables()

    if finder.tables:
        table = finder.tables[0]

        structured_rows = table.extract()
        markdown_table = table.to_markdown()

        table_record = {
            "page_number": 1,
            "row_count": table.row_count,
            "column_count": table.col_count,
            "bbox": table.bbox,
            "rows": structured_rows,
            "markdown": markdown_table,
        }

        print("Structured rows:")
        for row in table_record["rows"]:
            print(row)

        print("\nMarkdown:")
        print(table_record["markdown"])

Structured rows:
['Location', 'Nightlylimit', 'Notes']
['Lagos', '₦120000\n,', 'Standardbusinessaccommodation']
['Abuja', '₦110000\n,', 'Standardbusinessaccommodation']
['PortHarcourt', '₦100000\n,', 'Standardbusinessaccommodation']
['International', 'Actualreasonablecost', 'Departmentapprovalrequired']

Markdown:
|Location|Nightlylimit|Notes|
|---|---|---|
|Lagos|₦120,000|Standard business accommodation|
|Abuja|₦110,000|Standard business accommodation|
|Port Harcourt|₦100,000|Standard business accommodation|
|International|Actual reasonable cost|Department approval required|




This is a much stronger foundation for later chunking.

We can use the Markdown representation for retrieval while retaining the structured representation for validation, exact extraction, or downstream application logic.

## 15. HTML structured content

HTML is another important structured format.

Unlike a PDF, HTML exposes semantic elements directly.

Inspect the actual support knowledge-base document.

In [15]:
from bs4 import BeautifulSoup

html = HTML_PATH.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

article = soup.find("article")

print("Article found:", article is not None)

if article:
    print("Document ID:", article.get("data-document-id"))

    for heading in article.find_all(["h1", "h2", "h3"]):
        print(heading.name, "->", heading.get_text(" ", strip=True))

Article found: True
Document ID: KB-REF-2026-07
h1 -> Refunds and Cancellations
h2 -> Refund eligibility
h2 -> Processing time
h2 -> Contact support


## 16. HTML tables

First check whether the actual HTML document contains tables.

We should inspect the source instead of assuming that every structured document contains one.

In [16]:
html_tables = soup.find_all("table")

print("HTML tables:", len(html_tables))

for i, table in enumerate(html_tables, start=1):
    print(f"Table {i}:")
    for row in table.find_all("tr"):
        print([
            cell.get_text(" ", strip=True)
            for cell in row.find_all(["th", "td"])
        ])

HTML tables: 0


A zero count is a valid result.

This is important in ingestion engineering: **absence of a structure is an observation, not a failure.**

Our HTML asset contains useful semantic structure even if it does not provide a table.

## 17. Markdown tables

Markdown can represent tables directly.

Inspect the actual Markdown source rather than manufacturing a table for the lesson.

In [17]:
markdown_text = MD_PATH.read_text(encoding="utf-8")

for line in markdown_text.splitlines():
    if "|" in line:
        print(line)

If a Markdown document contains a table, it can often be parsed directly.

But again, we should distinguish:

```text
Markdown table syntax
        ↓
textual representation of structure
```

from a native structured table object.

The downstream requirements determine whether further parsing is worthwhile.


## 18. Table provenance

A table needs provenance just like ordinary text.

A useful record includes:

```text
document_id
page_number
table_index
bbox
source_filename
extraction_method
parser_version
```

This allows us to trace a retrieved table or table-derived chunk back to the source.

In [18]:
from hashlib import sha256
from importlib.metadata import version, PackageNotFoundError

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

def file_hash(path):
    digest = sha256()
    with path.open("rb") as f:
        while chunk := f.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

table_provenance = {
    "document_id": file_hash(PDF_PATH),
    "source_filename": PDF_PATH.name,
    "page_number": 1,
    "table_index": 1,
    "extraction_method": "pymupdf.find_tables",
    "parser_version": package_version("pymupdf")
}

print(table_provenance)

{'document_id': 'e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9', 'source_filename': 'employee_travel_policy.pdf', 'page_number': 1, 'table_index': 1, 'extraction_method': 'pymupdf.find_tables', 'parser_version': '1.28.2'}


This connects directly to Notebook 07.

**Metadata and provenance are not optional extras added after chunking.**

Structured content needs the same source lineage as ordinary text.

## 19. Table validation beyond row counts

Shape validation is only the beginning.

For a production pipeline, useful checks include:

- expected number of columns,
- non-empty headers,
- duplicate headers,
- unexpected empty cells,
- suspiciously long cells,
- numeric field validation,
- row count changes between extraction methods,
- and comparison against known source constraints.

The checks should be appropriate to the document type.

In [19]:
def validate_rows(rows):
    if not rows:
        return {
            "valid": False,
            "reason": "no_rows",
        }
    width = len(rows[0])

    empty_headers = [
        i for i, value in enumerate(rows[0])
        if not str(value or "").strip()
    ]

    inconsistent_rows = [
        i for i, row in enumerate(rows)
        if len(row) != width
    ]

    return {
        "valid": not empty_headers and not inconsistent_rows,
        "column_count": width,
        "empty_header_indexes": empty_headers,
        "inconsistent_row_indexes": inconsistent_rows,
    }

with pymupdf.open(PDF_PATH) as pdf:
    table = pdf[0].find_tables().tables[0]
    rows = table.extract()

print(validate_rows(rows))

{'valid': True, 'column_count': 3, 'empty_header_indexes': [], 'inconsistent_row_indexes': []}


## 20. When should a table become chunks?

There is no universal answer.

### Retrieval-oriented table

If users ask questions such as:

> What is the accommodation limit in Abuja?

A table can be represented as Markdown or row-oriented text and indexed for retrieval.

### Application-oriented table

If an application needs:

```text
location → nightly_limit
```

then structured storage may be more appropriate.

### High-value complex table

For financial, legal, scientific, or operational tables, it may be useful to retain:

```text
original structured table
+
retrieval representation
```

This gives the application both exact data and searchable context.

## 22. Do not blindly flatten every table

A common ingestion mistake is:

```text
PDF
 ↓
all text
 ↓
chunk
```

even when the document contains important tables.

A better mental model is:

```text
Document
 ├── prose
 ├── headings
 ├── lists
 ├── tables
 └── other structured regions
```

Each region can then be represented appropriately before chunking.


## 23. Production architecture for structured documents

A production ingestion path can look like:

```text
                    Document
                       │
                       ▼
                 Format detection
                       │
        ┌──────────────┼──────────────┐
        ▼              ▼              ▼
      PDF             DOCX           HTML
        │              │              │
        ▼              ▼              ▼
   PyMuPDF table   python-docx     semantic HTML
   extraction       tables          extraction
        │              │              │
        └──────────────┼──────────────┘
                       ▼
             Canonical structured
                  representation
                       │
              ┌────────┴────────┐
              ▼                 ▼
        structured data     RAG text
              │                 │
              └────────┬────────┘
                       ▼
                    chunking
                       │
                       ▼
                   indexing
```

The canonical representation becomes especially valuable because different parsers can feed the same downstream pipeline.

## 24. Extraction strategy: do not over-process

Structured extraction has a cost.

Do not automatically run expensive table detection against every page of every document if your corpus contains mostly prose.

Instead, use document profiling and corpus measurements to decide where specialized extraction is worthwhile.

The right strategy depends on:

- document types,
- table frequency,
- table complexity,
- extraction latency,
- failure rate,
- and the value of structured information.

## 25. Quality checks before indexing

For every extracted table, consider recording:

```text
detected: yes/no
rows
columns
headers
empty_cells
extraction_method
parser_version
provenance
validation_status
```

This lets ingestion monitoring answer questions such as:

> Are table extraction failures increasing after a parser upgrade?

That is much more useful than discovering months later that users cannot retrieve information from tables.

## 26. A compact structured-content record

Here is a conceptual record for a table extracted from the real PDF.

It deliberately keeps structured data and retrieval text separate.

In [20]:
import json

with pymupdf.open(PDF_PATH) as pdf:
    page = pdf[0]
    finder = page.find_tables()
    table = finder.tables[0]

    structured_record = {
        "type": "table",
        "provenance": {
            "source_filename": PDF_PATH.name,
            "page_number": 1,
            "table_index": 1,
            "parser": "pymupdf",
            "parser_version": package_version("pymupdf"),
            "document_id": file_hash(PDF_PATH),
        },
        "structure": {
            "rows": table.row_count,
            "columns": table.col_count,
            "bbox": table.bbox,
        },
        "data": table.extract(),
        "retrieval_text": table.to_markdown(),
    }

print(json.dumps(structured_record, indent=2, default=str)[:7000])

{
  "type": "table",
  "provenance": {
    "source_filename": "employee_travel_policy.pdf",
    "page_number": 1,
    "table_index": 1,
    "parser": "pymupdf",
    "parser_version": "1.28.2",
    "document_id": "e269b160c4cb9ae314a3efadf7285d33829c6d661abb81ea09b95d107b10c9a9"
  },
  "structure": {
    "rows": 5,
    "columns": 3,
    "bbox": [
      60.0,
      241.99996948242188,
      530.0,
      366.9999694824219
    ]
  },
  "data": [
    [
      "Location",
      "Nightlylimit",
      "Notes"
    ],
    [
      "Lagos",
      "\u20a6120000\n,",
      "Standardbusinessaccommodation"
    ],
    [
      "Abuja",
      "\u20a6110000\n,",
      "Standardbusinessaccommodation"
    ],
    [
      "PortHarcourt",
      "\u20a6100000\n,",
      "Standardbusinessaccommodation"
    ],
    [
      "International",
      "Actualreasonablecost",
      "Departmentapprovalrequired"
    ]
  ],
  "retrieval_text": "|Location|Nightlylimit|Notes|\n|---|---|---|\n|Lagos|\u20a6120,000|Standard busin

## 26. Key takeaways

1. **Tables are structured information, not just formatted text.**
2. **Preserve structure as early as possible.**
3. **DOCX exposes tables directly through its document structure.**
4. **PDF table extraction requires layout analysis and should be validated.**
5. **PyMuPDF provides low-level table control and geometry.**
6. **PyMuPDF4LLM provides a convenient RAG-oriented textual representation.**
7. **Markdown is useful, but it can still lose complex table semantics.**
8. **For important tables, retain both structured and retrieval representations.**
9. **Table provenance should include document and location information.**
10. **Row-level retrieval should preserve headers and table context.**
11. **Do not run expensive structured extraction blindly; measure your corpus first.**
12. **A canonical structured representation makes downstream processing more reliable.**

## What's Next?

## General Document Parsers

We have now examined format-specific parsing for PDFs and structured documents. Next we move to general-purpose document parsing frameworks, focusing on Unstructured and Docling and how they provide a common document-element representation across different file types.